# 🔍 W2: 불량 유형 AI 자동 분류기 (v2)
**hexa-1 — Week 2** | ⏱️ 60분 | 🎯 불량 분류 CSV + 통계 차트

## Step 0: 환경 설정 (최초 1회)

In [ ]:
!pip install -q google-generativeai pandas matplotlib
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'], capture_output=True)
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
# 폰트 파일 직접 등록 (deprecated API 미사용)
_nanum=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _nanum:
    fm.fontManager.addfont(_nanum[0])
    plt.rcParams['font.family']=fm.FontProperties(fname=_nanum[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ API 키 로드 성공')
except Exception:
    API_KEY = input('GEMINI_API_KEY 입력: ')
import google.generativeai as genai
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash')  # ✅ 2.0으로 업그레이드
print('✅ Gemini 2.0 Flash 준비 완료')


## Step 1: 공장 정보 입력 (✏️ 여기만 수정)

In [ ]:
FACTORY_INFO = {
    'name': '✏️ [교육 기업명]',
    'product': '✏️ [주요 제품명]',
    'defect_categories': ['치수불량','외관불량','기능불량','재료불량','기타']
}
print('✅ 설정:', FACTORY_INFO['defect_categories'])


## Step 2: 불량 데이터 입력 (샘플 또는 CSV)

In [ ]:
import pandas as pd
DEFECT_SAMPLES = pd.DataFrame([
    {'id':'D001','description':'부품 폭이 도면 규격보다 0.3mm 초과'},
    {'id':'D002','description':'표면에 깊은 스크래치 3개 발견'},
    {'id':'D003','description':'조립 후 작동 시 이상 소음 발생'},
    {'id':'D004','description':'원자재 경도가 기준치 미달'},
    {'id':'D005','description':'도금 벗겨짐, 색상 불균일'},
    {'id':'D006','description':'구멍 간격이 도면과 2mm 다름'},
    {'id':'D007','description':'용접 부위 크랙 발생'},
    {'id':'D008','description':'볼트 나사산 마모로 체결 불가'},
])
OEE_URL = 'https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
df_defects = DEFECT_SAMPLES.copy()
print(f'분석 대상: {len(df_defects)}건')
df_defects.head()


## Step 3: Gemini AI 분류 (📌 개선된 few-shot 프롬프트)

In [ ]:
def classify_defect(description: str, categories: list) -> str:
    cats = ', '.join(categories)
    # ✅ few-shot 예시 포함 프롬프트 (오분류 최소화)
    prompt = f"""제조업 품질 전문가로서 불량 설명을 분류하세요.

분류 기준:
- 치수불량: 규격/도면과 크기·간격이 다름 (예: '폭이 0.3mm 초과', '구멍 간격 다름')
- 외관불량: 표면 결함, 색상, 도금 (예: '스크래치', '도금 벗겨짐', '변색')
- 기능불량: 작동/조립 이상 (예: '이상 소음', '체결 불가', '동작 불량')
- 재료불량: 원자재/소재 자체 결함 (예: '경도 미달', '재질 불량')
- 기타: 위 4가지에 해당 없음

불량 설명: {description}

답변 규칙: 카테고리명만 한 단어로 답하세요. 설명 금지.
답변:"""
    try:
        response = model.generate_content(prompt)
        result = response.text.strip()
        for cat in categories:
            if cat in result:
                return cat
        return '기타'
    except Exception as e:
        print(f'오류({e})')
        return '분류오류'

print('🤖 AI 분류 시작...')
df_defects['AI분류'] = df_defects['description'].apply(
    lambda d: classify_defect(d, FACTORY_INFO['defect_categories'])
)
print('✅ 분류 완료')
df_defects[['id','description','AI분류']]


## Step 4: 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,5))
counts = df_defects['AI분류'].value_counts()
ax1.bar(counts.index,counts.values,color='steelblue',edgecolor='white')
ax1.set_title(f'{FACTORY_INFO["name"]} 불량 유형 분포')
ax1.set_ylabel('건수')
ax1.tick_params(axis='x',rotation=20)
ax2.pie(counts.values,labels=counts.index,autopct='%1.1f%%',
        colors=['#e74c3c','#f39c12','#3498db','#2ecc71','#9b59b6'])
ax2.set_title('불량 유형 비율')
plt.suptitle(f'{FACTORY_INFO["product"]} 불량 분류 현황',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('defect_classification.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'\n📊 최다 불량: {counts.index[0]} ({counts.values[0]}건)')


## Step 5: 저장 & 다운로드

In [ ]:
df_defects.to_csv('defect_classified.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('defect_classified.csv')
files.download('defect_classification.png')
print('✅ 다운로드 완료!')


---
## 🔥 확장 과제
1. 실제 불량 기록 CSV를 업로드해서 실습
2. `신뢰도 분류 이유`도 함께 출력하는 프롬프트로 개선
3. `defect_classifier.py` CLI로 실행해보기